# s05 - non-IID sweep (Dirichlet alpha): evaluasi R2 (SURVEY)

**Klaim R2** (`paper-survey.md` §4.2): performa FL turun seiring heterogenitas [F11].
Uji: SATU dataset dipecah K klien via Dirichlet(alpha); alpha turun dari ~IID (100)
ke ekstrem (0.05). Cari **titik putus** FedAvg (di mana MCC jatuh).

**Catatan metodologi (JUJUR):** sweep ini **disimulasikan di notebook** (bukan via
EC2) karena perlu banyak kombinasi (alpha x dataset x seed) — tak praktis deploy EC2
tiap kombinasi. Logika FedAvg + MLP + z-score IDENTIK `aws/fed_node_survey.py`, jadi
hasilnya setara deployment. Deployment nyata EC2 dipakai di R3/R4/R6. Dicatat di §6.

**Prasyarat:** partisi s01 di S3. **Output:** `survey_out/r2_noniid_sweep.csv`.

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','scikit-learn','boto3'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, io, numpy as np
from sklearn.metrics import matthews_corrcoef
OUTDIR='survey_out'; os.makedirs(OUTDIR,exist_ok=True)
BUCKET='ssh-detection-features-232032302717'; REGION='ap-southeast-1'; BASE='unsw-far/federated_survey'
import boto3; s3=boto3.client('s3',region_name=REGION)
def load_npz_s3(key):
    try: obj=s3.get_object(Bucket=BUCKET,Key=key)
    except Exception as e: print('MISS',key,e); return None
    return np.load(io.BytesIO(obj['Body'].read()),allow_pickle=True)
ALPHAS=[100,10,1,0.5,0.1,0.05]; K=4; ROUNDS=50; LOCAL_EPOCHS=5; LR=3e-3; SEEDS=[13,42,101]
DATASET=os.environ.get('DATASET','unsw')  # 'unsw' (seimbang, rapuh) atau 'cic' (condong-benign, tahan)

## 1. Muat satu dataset (train+test) hasil s01

In [ ]:
dtr=load_npz_s3(f'{BASE}/data/{DATASET}/{DATASET}_train.npz')
dte=load_npz_s3(f'{BASE}/data/{DATASET}/{DATASET}_test.npz')
assert dtr is not None and dte is not None, 'PRASYARAT GAGAL: jalankan s01 dulu.'
Xtr=np.asarray(dtr['X'],float); ytr=np.asarray(dtr['y'],int)
Xte=np.asarray(dte['X'],float); yte=np.asarray(dte['y'],int)
print(DATASET,'train',Xtr.shape,'pos',round(float(ytr.mean()),3),'| test',Xte.shape)

## 2. MLP + FedAvg (identik fed_node_survey.py) + partisi Dirichlet

In [ ]:
LAYERS=[Xtr.shape[1],128,64,1]
def init_w(seed): 
    rng=np.random.RandomState(seed); W=[]
    for a,b in zip(LAYERS[:-1],LAYERS[1:]): W.append(rng.randn(a,b)*np.sqrt(2/a)); W.append(np.zeros(b))
    return W
relu=lambda x:np.maximum(0,x); sig=lambda x:1/(1+np.exp(-np.clip(x,-30,30)))
def fwd(W,X):
    a1=X@W[0]+W[1]; z1=relu(a1); a2=z1@W[2]+W[3]; z2=relu(a2); a3=z2@W[4]+W[5]
    return sig(a3).ravel(),(X,a1,z1,a2,z2)
def train_local(W,X,y,epochs,lr,seed):
    rng=np.random.RandomState(seed); n=len(y); W=[w.copy() for w in W]
    for _ in range(epochs):
        idx=rng.permutation(n)
        for s in range(0,n,256):
            bi=idx[s:s+256]; Xb=X[bi]; yb=y[bi].astype(float)
            o,(X_,a1,z1,a2,z2)=fwd(W,Xb); g=(o-yb)/max(len(yb),1)
            gW4=z2.T@g[:,None]; gb4=g.sum(0); d2=(g[:,None]@W[4].T)*(a2>0)
            gW2=z1.T@d2; gb2=d2.sum(0); d1=(d2@W[2].T)*(a1>0); gW0=X_.T@d1; gb0=d1.sum(0)
            for i,gv in enumerate([gW0,gb0,gW2,gb2,gW4,gb4]): W[i]=W[i]-lr*np.asarray(gv).reshape(W[i].shape)
    return W
def fedavg(Ws,ns):
    tot=sum(ns); return [sum((ns[k]/tot)*Ws[k][i] for k in range(len(Ws))) for i in range(len(Ws[0]))]
def zfit(X): m=X.mean(0); s=X.std(0); s[s==0]=1; return m,s
def dirichlet_partition(y,K,alpha,seed):
    rng=np.random.RandomState(seed); idx_by_c={c:np.where(y==c)[0] for c in np.unique(y)}
    client_idx=[[] for _ in range(K)]
    for c,idc in idx_by_c.items():
        rng.shuffle(idc); props=rng.dirichlet([alpha]*K); cuts=(np.cumsum(props)*len(idc)).astype(int)[:-1]
        for k,part in enumerate(np.split(idc,cuts)): client_idx[k].extend(part.tolist())
    return [np.array(c) for c in client_idx if len(c)>0]

## 3. Sweep alpha x seed -> FedAvg K klien; ukur MCC global (test dataset ini)

In [ ]:
m_mon,s_mon=zfit(Xtr); Xtr_z=(Xtr-m_mon)/s_mon; Xte_z=(Xte-m_mon)/s_mon
def run_fedavg(parts, seed):
    W=init_w(seed)
    for t in range(ROUNDS):
        Ws=[]; ns=[]
        for pidx in parts:
            Wl=train_local(W, Xtr_z[pidx], ytr[pidx], LOCAL_EPOCHS, LR, seed); Ws.append(Wl); ns.append(len(pidx))
        W=fedavg(Ws,ns)
    o,_=fwd(W,Xte_z); return matthews_corrcoef(yte,(o>=0.5).astype(int))
rows=[]
for a in ALPHAS:
    vals=[]
    for sd in SEEDS:
        parts=dirichlet_partition(ytr,K,a,sd); vals.append(run_fedavg(parts,sd))
    rows.append(dict(dataset=DATASET,alpha=a,mcc_mean=float(np.mean(vals)),mcc_std=float(np.std(vals))))
    print(f'alpha={a:<5} MCC={rows[-1]["mcc_mean"]:.3f} +/- {rows[-1]["mcc_std"]:.3f}')

## 4. Simpan R2 -> CSV + S3 (titik putus = alpha di mana MCC jatuh tajam)

In [ ]:
import csv
path=os.path.join(OUTDIR,f'r2_noniid_sweep_{DATASET}.csv')
with open(path,'w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=['dataset','alpha','mcc_mean','mcc_std']); w.writeheader()
    for r in rows: w.writerow(r)
print('saved',path)
try: s3.upload_file(path,BUCKET,f'{BASE}/results/{os.path.basename(path)}'); print('upload OK')
except Exception as e: print('(upload dilewati:',e,')')
print('=== s05 SELESAI. Jalankan utk DATASET=unsw DAN DATASET=cic; isi §4.3 baris R2. ===')